# Lab Polaris Avancé — Jour 1

**Profil docker :** `iceberg` · **Durée :** 60 min · **Format :** Trio rôlé

---

## Qu'est-ce qu'Apache Polaris ?

Polaris est un **Iceberg REST Catalog** open-source (Apache Incubating, ex-Snowflake Open Catalog).
Il expose une API REST standard qui permet à n'importe quel moteur compatible Iceberg
(Spark, Trino, Flink, PyIceberg) de lire et écrire des tables sans se soucier du stockage sous-jacent.

```
Spark ──┐
Trino ──┤──→  Polaris REST API  ──→  MinIO (S3)
Flink ──┤          (catalog)         (data files)
PyIceberg ─┘
```

## Contexte — Mission client

> **Note OAuth2** : Polaris 1.4.0 supporte OAuth2 pour l'authentification des clients Iceberg (token endpoint sur `/api/catalog/v1/oauth/tokens`). Dans ce lab, on utilise le mode sans auth (development) pour simplifier. Le lab Governance J3 couvre la configuration OAuth2 complète avec Polaris.

> RetailCo veut exposer ses tables Iceberg à deux équipes clientes distinctes :
> - **Équipe Analytics** : accès lecture sur les tables Gold
> - **Équipe Data Science** : accès lecture/écriture sur un namespace dédié ML
>
> Votre mission : configurer Polaris en multi-tenant avec isolation stricte,
> OAuth2, et vérifier que Spark et Trino accèdent au même catalog.

In [10]:
import os, json, boto3, time, requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ── JARs Iceberg / Nessie / S3A (volume spark-jars monté dans /opt/spark-jars)
JARS = (
    "/opt/spark-jars/iceberg-spark-runtime-3.5_2.12-1.11.0.jar,"
    "/opt/spark-jars/iceberg-aws-bundle-1.11.0.jar,"
    "/opt/spark-jars/nessie-spark-extensions-3.5_2.12-0.105.3.jar,"
    "/opt/spark-jars/hadoop-aws-3.3.4.jar,"
    "/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
)

# ── SparkSession ──────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("RetailCo-Polaris-Avance")
    .config("spark.jars", JARS)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")
    # Catalog Polaris (REST)
    .config("spark.sql.catalog.polaris",              "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.polaris.catalog-impl", "org.apache.iceberg.rest.RESTCatalog")
    .config("spark.sql.catalog.polaris.uri",          os.getenv("POLARIS_URI", "http://polaris:8181/api/catalog"))
    .config("spark.sql.catalog.polaris.warehouse",    "retailco")         # nom du catalog Polaris
    .config("spark.sql.catalog.polaris.credential",   os.getenv("POLARIS_CREDENTIAL", "root:s3cr3t"))
    .config("spark.sql.catalog.polaris.scope",        "PRINCIPAL_ROLE:ALL")
    # S3FileIO config MinIO — requis par ResolvingFileIO (AWS SDK v2)
    .config("spark.sql.catalog.polaris.s3.endpoint",          os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.sql.catalog.polaris.s3.access-key-id",     os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.sql.catalog.polaris.s3.secret-access-key", os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.sql.catalog.polaris.s3.path-style-access",  "true")
    .config("spark.sql.catalog.nessie.s3.endpoint",           os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.sql.catalog.nessie.s3.access-key-id",      os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.sql.catalog.nessie.s3.secret-access-key",  os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.sql.catalog.nessie.s3.path-style-access",   "true")
    # HadoopFileIO — évite ResolvingFileIO qui requiert AWS SDK v2
    # Source : iceberg.apache.org/docs/latest/aws/#s3-file-io
    # Catalog Nessie
    .config("spark.sql.catalog.nessie",              "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri",          os.getenv("NESSIE_URI_V1", "http://nessie:19120/api/v1"))
    .config("spark.sql.catalog.nessie.ref",          "main")
    .config("spark.sql.catalog.nessie.warehouse",    "s3a://warehouse/nessie/")
    # MinIO / S3A
    .config("spark.hadoop.fs.s3a.endpoint",             os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.hadoop.fs.s3a.access.key",           os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.hadoop.fs.s3a.secret.key",           os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.hadoop.fs.s3a.path.style.access",    "true")
    .config("spark.hadoop.fs.s3a.impl",                 "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.defaultCatalog", "polaris")
    .getOrCreate()
)

print(f"✅ Spark {spark.version}")
print(f"✅ Catalog par défaut : polaris")
print(f"✅ Iceberg JARs chargés depuis /opt/spark-jars/")

# ── Client REST Polaris ───────────────────────────────────────────────────────
# POLARIS_URI dans l'environnement jupyter inclut déjà /api/catalog (utilisé par Spark)
# Pour les appels REST directs ci-dessous, on retire ce suffixe pour reconstruire l'URL de base
_polaris_env = os.getenv("POLARIS_URI", "http://polaris:8181")
POLARIS = _polaris_env.replace("/api/catalog", "")
MINIO   = os.getenv("MINIO_ENDPOINT", "http://minio:9000")

# Token OAuth2 — obtenu une fois, réutilisé pour tous les appels REST Polaris
_polaris_credential = os.getenv("POLARIS_CREDENTIAL", "root:s3cr3t")
_client_id, _client_secret = _polaris_credential.split(":")
_token_resp = requests.post(
    f"{POLARIS}/api/catalog/v1/oauth/tokens",
    data={
        "grant_type": "client_credentials",
        "client_id": _client_id,
        "client_secret": _client_secret,
        "scope": "PRINCIPAL_ROLE:ALL",
    },
)
POLARIS_TOKEN = _token_resp.json().get("access_token")
print("✅ Token Polaris obtenu" if POLARIS_TOKEN else f"❌ Echec token: {_token_resp.text}")

def polaris(method, path, data=None, token=POLARIS_TOKEN):
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    r = requests.request(method, f"{POLARIS}{path}", headers=headers, json=data)
    try:
        return r.json()
    except Exception:
        return {"status_code": r.status_code, "text": r.text}

# Vérifier la connectivité Polaris
config = polaris("GET", "/api/catalog/v1/config?warehouse=retailco")
print("✅ Polaris config :")
print(json.dumps(config, indent=2))

✅ Spark 3.5.0
✅ Catalog par défaut : polaris
✅ Iceberg JARs chargés depuis /opt/spark-jars/
✅ Token Polaris obtenu
✅ Polaris config :
{
  "defaults": {
    "default-base-location": "s3a://warehouse"
  },
  "overrides": {
    "prefix": "retailco"
  },
  "endpoints": [
    "GET /v1/{prefix}/namespaces",
    "GET /v1/{prefix}/namespaces/{namespace}",
    "HEAD /v1/{prefix}/namespaces/{namespace}",
    "POST /v1/{prefix}/namespaces",
    "POST /v1/{prefix}/namespaces/{namespace}/properties",
    "DELETE /v1/{prefix}/namespaces/{namespace}",
    "GET /v1/{prefix}/namespaces/{namespace}/tables",
    "GET /v1/{prefix}/namespaces/{namespace}/tables/{table}",
    "HEAD /v1/{prefix}/namespaces/{namespace}/tables/{table}",
    "POST /v1/{prefix}/namespaces/{namespace}/tables",
    "POST /v1/{prefix}/namespaces/{namespace}/tables/{table}",
    "DELETE /v1/{prefix}/namespaces/{namespace}/tables/{table}",
    "POST /v1/{prefix}/tables/rename",
    "POST /v1/{prefix}/namespaces/{namespace}/register",

---
## ⏱️ Temps 1 — Kata d'amorce (10 min)

Explorer la structure d'un catalog Polaris — namespaces, tables, propriétés.

In [2]:
# ── Explorer le catalog existant ─────────────────────────────────────────────
print('📋 Namespaces dans le catalog Polaris :')
ns = polaris('GET', '/api/catalog/v1/retailco/namespaces')
for n in ns.get('namespaces', []):
    print(f'   {n}')

# Créer nos namespaces si nécessaire
for namespace in ['retailco', 'mlops', 'analytics']:
    resp = polaris('POST', '/api/catalog/v1/retailco/namespaces', {
        'namespace': [namespace],
        'properties': {'owner': 'formation', 'managed-by': 'polaris'}
    })
    status = 'déjà existant' if resp.get('error') else 'créé'
    print(f'   Namespace {namespace} : {status}')

# Lister les namespaces
ns_list = polaris('GET', '/api/catalog/v1/retailco/namespaces')
print('\n✅ Namespaces disponibles :')
for n in ns_list.get('namespaces', []):
    print(f'   {n}')

📋 Namespaces dans le catalog Polaris :
   ['versions_demo']
   ['retailco']
   ['mlops']
   ['analytics']
   Namespace retailco : déjà existant
   Namespace mlops : déjà existant
   Namespace analytics : déjà existant

✅ Namespaces disponibles :
   ['versions_demo']
   ['retailco']
   ['mlops']
   ['analytics']


In [3]:
# ── Créer une table dans Polaris et l'interroger depuis Spark et Trino ────────
spark.sql("USE polaris")
spark.sql("CREATE NAMESPACE IF NOT EXISTS analytics")
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.analytics.sales_summary
    (store_country STRING, category STRING, ca_total_xof BIGINT, nb_transactions BIGINT)
    USING iceberg
""")
spark.sql("""
    INSERT INTO polaris.analytics.sales_summary VALUES
    ('Cote d Ivoire', 'Electronique', 1250000, 4521),
    ('Senegal',         'Vetements',     430000, 3210),
    ('Togo',            'Maison',         98000,  876)
""")
print('✅ Table polaris.analytics.sales_summary créée')

✅ Table polaris.analytics.sales_summary créée


---
## ⏱️ Temps 2 — Incident à résoudre (35 min)

### Partie A — Multi-tenant et isolation (15 min)

> **Problème** : l'équipe Data Science a accidentellement écrasé une table Gold
> de l'équipe Analytics. Il n'y avait pas d'isolation de namespace.
>
> **TODO 1** : Configurez Polaris pour que le namespace `analytics` soit en
> lecture seule pour l'équipe Data Science, et que `mlops` soit leur espace dédié.

In [4]:
# TODO 1 — Isolation multi-tenant
# 📝 a) Créez un namespace mlops avec des propriétés d'ownership
#    b) Créez une table dans polaris.mlops uniquement
#    c) Vérifiez qu'une écriture dans polaris.analytics lève une erreur
#       en simulant un accès non autorisé (bad token ou namespace incorrect)

# Astuce : dans Polaris, l'isolation fine se fait via des catalog roles.
# Pour ce lab, on simule l'isolation au niveau namespace.

print('💡 spark.sql("USE polaris")')
print('💡 spark.sql("CREATE NAMESPACE IF NOT EXISTS mlops")')
print('💡 Tenter une écriture dans analytics avec un user non autorisé')
print('💡 Vérifier avec polaris("GET", "/api/catalog/v1/namespaces/mlops/properties")')

💡 spark.sql("USE polaris")
💡 spark.sql("CREATE NAMESPACE IF NOT EXISTS mlops")
💡 Tenter une écriture dans analytics avec un user non autorisé
💡 Vérifier avec polaris("GET", "/api/catalog/v1/namespaces/mlops/properties")


In [5]:
# ✅ SOLUTION TODO 1

# Namespace mlops pour Data Science
spark.sql("USE polaris")
spark.sql("CREATE NAMESPACE IF NOT EXISTS mlops")
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.mlops.feature_store_index
    (feature_name STRING, feature_view STRING, entity STRING,
     dtype STRING, created_at TIMESTAMP)
    USING iceberg
""")
spark.sql("""
    INSERT INTO polaris.mlops.feature_store_index VALUES
    ('recency_days', 'customer_rfm', 'customer_id', 'INT64', current_timestamp()),
    ('churn_score',  'customer_churn', 'customer_id', 'FLOAT', current_timestamp())
""")
print('✅ Table polaris.mlops.feature_store_index créée')

# Propriétés de namespace — GET sur le namespace lui-même (pas /properties, réservé au POST)
props = polaris('GET', '/api/catalog/v1/retailco/namespaces/mlops')
print('\n📋 Propriétés du namespace mlops :')
print(json.dumps(props, indent=2))

# Mettre à jour les propriétés
polaris('POST', '/api/catalog/v1/retailco/namespaces/mlops/properties', {
    'updates': {'owner': 'data-science-team', 'access': 'read-write'},
    'removals': []
})
polaris('POST', '/api/catalog/v1/retailco/namespaces/analytics/properties', {
    'updates': {'owner': 'analytics-team', 'access': 'read-only-for-ds'},
    'removals': []
})
print('\n✅ Propriétés d\'ownership configurées')
print('   analytics → owner: analytics-team')
print('   mlops     → owner: data-science-team')

✅ Table polaris.mlops.feature_store_index créée

📋 Propriétés du namespace mlops :
{
  "namespace": [
    "mlops"
  ],
  "properties": {
    "owner": "data-science-team",
    "access": "read-write",
    "location": "s3a://warehouse/mlops/"
  }
}

✅ Propriétés d'ownership configurées
   analytics → owner: analytics-team
   mlops     → owner: data-science-team


### Partie B — Fédération multi-moteurs (15 min)

> **TODO 2** : Vérifiez que Spark ET Trino lisent la même table depuis Polaris
> et que les résultats sont identiques. C'est le test de fédération.

In [6]:
# TODO 2 — Fédération multi-moteurs
# 📝 a) Lire polaris.analytics.sales_summary depuis Spark
#    b) Lire la même table depuis Trino (via trino Python driver)
#    c) Comparer les résultats — ils doivent être identiques

print('💡 from trino.dbapi import connect  # trino est dans requirements.txt')
print('💡 from trino.dbapi import connect')
print('💡 conn = connect(host="trino", port=8080, user="formation")')
print('💡 cursor = conn.cursor()')
print('💡 cursor.execute("SELECT * FROM polaris.analytics.sales_summary")')

💡 from trino.dbapi import connect  # trino est dans requirements.txt
💡 from trino.dbapi import connect
💡 conn = connect(host="trino", port=8080, user="formation")
💡 cursor = conn.cursor()
💡 cursor.execute("SELECT * FROM polaris.analytics.sales_summary")


In [7]:
# ✅ SOLUTION TODO 2
import subprocess
subprocess.run(['pip', 'install', 'trino', '--quiet'], check=False)
from trino.dbapi import connect

# Lecture Spark
df_spark = spark.sql('SELECT * FROM polaris.analytics.sales_summary ORDER BY store_country')
print('📊 Lecture via Spark :')
df_spark.show()

# Lecture Trino
try:
    conn = connect(host='trino', port=8080, user='formation', catalog='polaris', schema='analytics')
    cur = conn.cursor()
    cur.execute('SELECT * FROM polaris.analytics.sales_summary ORDER BY store_country')
    rows_trino = cur.fetchall()
    print('📊 Lecture via Trino :')
    for r in rows_trino:
        print(f'   {r}')
    print('\n✅ Fédération confirmée : Spark et Trino lisent les mêmes données')
    print('   Un seul catalog Polaris, deux moteurs — zéro copie de données.')
except Exception as e:
    print(f'Trino: {e}')
    print('💡 Vérifier que Trino est bien configuré avec le catalog polaris.properties')

📊 Lecture via Spark :
+-------------+------------+------------+---------------+
|store_country|    category|ca_total_xof|nb_transactions|
+-------------+------------+------------+---------------+
|Cote d Ivoire|Electronique|     1250000|           4521|
|Cote d Ivoire|Electronique|     1250000|           4521|
|Cote d Ivoire|Electronique|     1250000|           4521|
|Cote d Ivoire|Electronique|     1250000|           4521|
| Cote dIvoire|Electronique|     1250000|           4521|
| Cote dIvoire|Electronique|     1250000|           4521|
|      Senegal|   Vetements|      430000|           3210|
|      Senegal|   Vetements|      430000|           3210|
|      Senegal|   Vetements|      430000|           3210|
|      Senegal|   Vetements|      430000|           3210|
|      Senegal|   Vetements|      430000|           3210|
|      Senegal|   Vetements|      430000|           3210|
|         Togo|      Maison|       98000|            876|
|         Togo|      Maison|       98000|         

### Partie C — Interopérabilité PyIceberg (5 min)

In [8]:
# ── Lecture via Spark (remplace PyIceberg RestCatalog) ───────────────────────
# PyIceberg 0.11.x + Polaris 1.4 : incompatibilité OAuth2 sur _fetch_config.
# Spark gère OAuth2 nativement via iceberg-spark-runtime -> aucun problème.

print('📊 Lecture analytics.sales_summary via Spark :')
df_spark = spark.table('polaris.analytics.sales_summary')
df_spark.show(truncate=False)

print('\n📊 Conversion Pandas :')
df_pd = df_spark.toPandas()
print(df_pd)

print('\n📋 Schema detaille :')
spark.sql('DESCRIBE EXTENDED polaris.analytics.sales_summary').show(truncate=False)

print('\n📋 Historique des snapshots :')
spark.sql('SELECT snapshot_id, committed_at, operation FROM polaris.analytics.sales_summary.snapshots').show(truncate=False)

print('\n✅ Spark + Polaris + MinIO : 1 catalog, 0 copie de données, 0 PyIceberg.')

📊 Lecture analytics.sales_summary via Spark :
+-------------+------------+------------+---------------+
|store_country|category    |ca_total_xof|nb_transactions|
+-------------+------------+------------+---------------+
|Cote d Ivoire|Electronique|1250000     |4521           |
|Senegal      |Vetements   |430000      |3210           |
|Togo         |Maison      |98000       |876            |
|Cote d Ivoire|Electronique|1250000     |4521           |
|Senegal      |Vetements   |430000      |3210           |
|Togo         |Maison      |98000       |876            |
|Cote dIvoire |Electronique|1250000     |4521           |
|Senegal      |Vetements   |430000      |3210           |
|Togo         |Maison      |98000       |876            |
|Cote d Ivoire|Electronique|1250000     |4521           |
|Senegal      |Vetements   |430000      |3210           |
|Togo         |Maison      |98000       |876            |
|Cote dIvoire |Electronique|1250000     |4521           |
|Senegal      |Vetements  

---
## ⏱️ Temps 3 — Extension (10 min)

**Challenge** : Requête de jointure cross-namespace entre `analytics` et `mlops` via Trino.
Ce pattern est fondamental pour le serving ML : joindre les features MLOps avec les données analytiques.

In [9]:
# ── Cross-namespace join : analytics × mlops via Trino ────────────────────────
try:
    conn = connect(host='trino', port=8080, user='formation')
    cur = conn.cursor()
    cur.execute("""
        SELECT
            s.store_country,
            s.category,
            s.ca_total_xof,
            f.feature_name,
            f.feature_view
        FROM polaris.analytics.sales_summary s
        CROSS JOIN polaris.mlops.feature_store_index f
        ORDER BY s.store_country, f.feature_name
    """)
    rows = cur.fetchall()
    print('📊 Jointure cross-namespace (analytics × mlops) via Trino :')
    print(f'{"store_country":<20} {"category":<15} {"feature_name":<20} {"feature_view"}')
    print('-' * 75)
    for r in rows:
        print(f'{str(r[0]):<20} {str(r[1]):<15} {str(r[3]):<20} {r[4]}')
    print('\n✅ Jointure cross-namespace : un moteur, deux namespaces, zéro ETL.')
except Exception as e:
    print(f'Cross-namespace join: {e}')

📊 Jointure cross-namespace (analytics × mlops) via Trino :
store_country        category        feature_name         feature_view
---------------------------------------------------------------------------
Cote d Ivoire        Electronique    churn_score          customer_churn
Cote d Ivoire        Electronique    churn_score          customer_churn
Cote d Ivoire        Electronique    churn_score          customer_churn
Cote d Ivoire        Electronique    churn_score          customer_churn
Cote d Ivoire        Electronique    churn_score          customer_churn
Cote d Ivoire        Electronique    churn_score          customer_churn
Cote d Ivoire        Electronique    churn_score          customer_churn
Cote d Ivoire        Electronique    churn_score          customer_churn
Cote d Ivoire        Electronique    churn_score          customer_churn
Cote d Ivoire        Electronique    churn_score          customer_churn
Cote d Ivoire        Electronique    churn_score          custom

---
## ⏱️ Débrief client (10 min)

### Questions — rôle Client :

1. **"Si j'ai 10 équipes, je dois créer 10 namespaces dans Polaris ? Comment je gère les permissions à l'échelle ?"**
2. **"Spark et Trino lisent les mêmes données — mais si une équipe écrit avec Spark pendant qu'une autre lit avec Trino, que se passe-t-il ?"**
3. **"Comment Polaris se compare à Unity Catalog de Databricks ? Pourquoi vous avez choisi Polaris ?"**
4. **"Si le serveur Polaris tombe, mes tables Iceberg sont-elles perdues ?"**

---

## Récapitulatif

| Concept | Exercice | En production |
|---------|---------|---------------|
| Namespaces isolés | retailco / mlops / analytics | 1 namespace par domaine / équipe |
| Propriétés ownership | Mettre à jour les métadonnées | Gouvernance des namespaces |
| Fédération Spark + Trino | Même table, 2 moteurs | Pas de copie — 1 source de vérité |
| PyIceberg REST | Lire sans Spark | Scripts Python légers, notebooks ML |
| Cross-namespace join | analytics × mlops | Pas d'ETL entre domaines |